# 전체 공개 데이터 1,029개 최종 학습

새 AudioSet AST에서 head 8 epoch, 마지막 블록 17 epoch를 고정 학습합니다. 모든 파일이 매 epoch에 참여했는지 기록합니다. 기존 평가 자료도 학습에 포함하므로 독립 자체 평가 점수는 계산하지 않습니다. 완료 후 제출 ZIP을 생성하고 별도 프로세스에서 오프라인 재현성을 확인합니다. A100은 최대 30분 후 해제됩니다. 이 노트북은 Colab 재현용이며 실행 결과는 실제 실행 후에만 생성됩니다.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, time, threading, traceback, zipfile
from google.colab import drive, runtime
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/DACON_236749')
WORK = Path('/content/dacon_full_refit')
WORK.mkdir(exist_ok=True)
START = time.time()
DEADLINE = START + 1680
RUN = DRIVE / ('full_refit_' + time.strftime('%Y%m%d_%H%M%S',time.gmtime(START)))
RUN.mkdir()
timer = threading.Timer(1680,runtime.unassign)
timer.daemon = True
timer.start()
print('RESULTS',RUN)

In [ ]:
%%writefile /content/dacon_full_refit/full_refit.py
"""Fixed-epoch final refit on the complete public corpus; no model selection."""
from collections import Counter
from contextlib import nullcontext
from dataclasses import dataclass
from pathlib import Path
import csv
import math
import random
import time

import numpy as np
import torch
from transformers import ASTFeatureExtractor

from ast_multitask import ASTMultiTask, compute_pos_weights, masked_multitask_loss
from graph_features import atomic_json, digest
from graph_scoring import LABEL_COLUMNS
from train_v2 import (CONFIGS, SEED, FrozenEmbeddingCache, _batch_inputs,
                      _frozen_logits, _json_hash, _optimizer, _targets,
                      _trainable_state, save_training_checkpoint)


@dataclass(frozen=True, slots=True)
class FullData:
    root: Path
    rows: tuple[dict[str, str], ...]
    manifest_hashes: dict[str, str]


@dataclass(frozen=True, slots=True)
class RefitJob:
    data: FullData
    output: Path
    device: torch.device
    epochs: int
    deadline: float


def read_full_data(root: Path) -> FullData:
    """Preserve original roles while explicitly admitting all four public roles."""
    rows = []
    hashes = {}
    allowed = {'dev_train', 'dev_select', 'test_generator', 'test_source'}
    for folder in ('development', 'locked_test'):
        path = root / folder / 'manifest.csv'
        hashes[folder] = digest(path)
        with path.open(encoding='utf-8-sig', newline='') as stream:
            rows.extend(csv.DictReader(stream))
    if not rows or len({r['ID'] for r in rows}) != len(rows):
        raise ValueError('full-data manifest is empty or has duplicate IDs')
    for row in rows:
        if row['split'] not in allowed:
            raise ValueError('unexpected public-data role')
        path = (root / row['path']).resolve()
        if not path.is_relative_to(root.resolve()) or not path.is_file():
            raise ValueError('missing audio or path outside public corpus')
        if any(row[c] not in {'', '0', '1'} for c in LABEL_COLUMNS):
            raise ValueError('invalid binary or unknown target')
    return FullData(root, tuple(sorted(rows, key=lambda r: r['ID'])), hashes)


def train_stage(model: ASTMultiTask, processor: ASTFeatureExtractor,
                job: RefitJob) -> dict:  # noqa: DICT_OK
    """Consume every ID once per epoch, saving the last epoch without holdout scoring."""
    name = 'AST_FROZEN' if model.mode == 'frozen' else 'AST_LAST_BLOCK'
    config = {**CONFIGS[name], 'epochs': job.epochs, 'seed': SEED,
              'manifest_hashes': job.data.manifest_hashes, 'purpose': 'full_public_data_refit'}
    job.output.mkdir(parents=True, exist_ok=False)
    atomic_json(job.output / 'config.json', config)
    model.to(job.device)
    optimizer = _optimizer(model, config)
    rows = list(job.data.rows)
    weights = compute_pos_weights(_targets(rows, job.device))
    cache = FrozenEmbeddingCache(job.output / 'cache', 'fresh-audioset-backbone',
                                  _json_hash(processor.to_dict())) if model.mode == 'frozen' else None
    exposures = Counter()
    history = []
    head_parameters = list(model.heads.parameters())
    backbone_parameters = [p for n, p in model.named_parameters()
                           if p.requires_grad and not n.startswith('heads.')]
    for epoch in range(1, job.epochs + 1):
        started = time.perf_counter()
        before = _trainable_state(model)
        model.train()
        optimizer.zero_grad(set_to_none=True)
        shuffled = list(rows)
        random.Random(SEED + epoch).shuffle(shuffled)
        consumed, losses, head_gradients, backbone_gradients = [], [], [], []
        batches = math.ceil(len(rows) / config['batch_size'])
        for index, start in enumerate(range(0, len(rows), config['batch_size'])):
            if time.time() > job.deadline:
                raise TimeoutError('full refit deadline; incomplete stage is not a final model')
            batch = shuffled[start:start + config['batch_size']]
            amp = torch.autocast('cuda', dtype=torch.bfloat16) if job.device.type == 'cuda' else nullcontext()
            with amp:
                logits = (_frozen_logits(model, processor, batch, job.data.root, cache, job.device)
                          if cache else model(_batch_inputs(batch, job.data.root, processor, 0, job.device)[0]))
                loss = masked_multitask_loss(logits, _targets(batch, job.device), weights)
            # Preserve the original accumulation rule, including the final partial group.
            (loss / config['accumulation']).backward()
            losses.append(float(loss.detach().cpu()))
            consumed.extend(row['ID'] for row in batch)
            if (index + 1) % config['accumulation'] == 0 or index + 1 == batches:
                head_gradients.append(float(torch.nn.utils.clip_grad_norm_(head_parameters, math.inf)))
                if backbone_parameters:
                    backbone_gradients.append(float(torch.nn.utils.clip_grad_norm_(backbone_parameters, math.inf)))
                norm = torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
                if not math.isfinite(float(norm)):
                    raise FloatingPointError('non-finite full-refit gradient')
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
            if (index + 1) % 20 == 0:
                print(f'FULL_BATCH {name} epoch={epoch} batch={index+1}/{batches}', flush=True)
        if Counter(consumed) != Counter(r['ID'] for r in rows):
            raise RuntimeError('epoch did not consume every public file exactly once')
        head_gradient = max(head_gradients)
        backbone_gradient = max(backbone_gradients, default=0.0)
        if head_gradient <= 0 or (backbone_parameters and backbone_gradient <= 0):
            raise FloatingPointError('intended trainable path received no gradient')
        exposures.update(consumed)
        after = _trainable_state(model)
        update = sum(float((value - before[n]).abs().sum()) for n, value in after.items())
        if update <= 0:
            raise FloatingPointError('no parameter update in completed epoch')
        report = {'epoch': epoch, 'mean_loss': float(np.mean(losses)),
                  'unique_files': len(set(consumed)), 'input_count': len(consumed),
                  'original_roles': dict(Counter(r['split'] for r in rows)),
                  'consumed_id_sha256': _json_hash(sorted(consumed)),
                  'head_gradient': head_gradient, 'backbone_gradient': backbone_gradient,
                  'absolute_parameter_update': update, 'seconds': time.perf_counter()-started}
        history.append(report)
        save_training_checkpoint(job.output / 'last.pt', model, optimizer, epoch, _json_hash(config), history)
        if job.device.type == 'mps':
            torch.save(torch.mps.get_rng_state(), job.output / 'mps_rng.pt')
        atomic_json(job.output / f'epoch_{epoch:03d}.json', report)
        print(f'FULL_EPOCH {name} {epoch}/{job.epochs} files={len(consumed)} loss={report["mean_loss"]:.6f}', flush=True)
    result = {'stage': name, 'epochs': history, 'exposures': dict(exposures),
              'epoch_count': job.epochs, 'config': config, 'selection': 'fixed_final_epoch',
              'independent_validation_score': None}
    atomic_json(job.output / 'report.json', result)
    return result


In [ ]:
%%writefile /content/dacon_full_refit/run_full_refit.py
"""Refit the selected AST recipe, then package and replay its final checkpoint."""
import argparse
import gc
import json
import os
from pathlib import Path
import shutil
import sys
import time
import zipfile
from collections import Counter

import numpy as np
import soundfile as sf
import torch

from ast_multitask import ASTMultiTask, export_checkpoint, load_processor
from full_refit import RefitJob, read_full_data, train_stage
from graph_features import atomic_json, digest
from graph_scoring import PREDICTION_COLUMNS
from package_submission import verify_packaged
from package_v2_submission import build_ast_submission
from run_experiment import write_csv
from train_v2 import seed_everything
from v2_inference import generate_submission


def main() -> None:
    """Only emit READY after all-data coverage and actual offline replay succeed."""
    parser = argparse.ArgumentParser()
    parser.add_argument('--assets', type=Path, required=True)
    parser.add_argument('--data', type=Path, required=True)
    parser.add_argument('--output', type=Path, required=True)
    parser.add_argument('--device', choices=('cuda', 'mps', 'cpu'), required=True)
    parser.add_argument('--seconds', type=int, default=3600)
    args = parser.parse_args()
    root = args.output.resolve()
    root.mkdir(parents=True, exist_ok=False)
    started = time.time()
    deadline = started + args.seconds
    device = torch.device(args.device)
    seed_everything()
    torch.set_num_threads(4)
    data = read_full_data(args.data.resolve())
    counts = dict(Counter(r['split'] for r in data.rows))
    assert counts == {'dev_train': 633, 'dev_select': 122, 'test_generator': 176, 'test_source': 98}
    assert data.manifest_hashes == {
        'development': '4fd72e3731de71c5ae4fccd4ca8a1e4d05f9e88a5f8143a60209e7cf9c2de01a',
        'locked_test': 'a11ae794de70f83c0a325f884d8ba920b4a1b3653a2936777a08a0be7c0c735d'}
    assert digest(args.assets / 'model.safetensors') == 'ae0c1e2ad4e1381d851fa9bf298ba13ebc9c5a914cdee2dbe427a6583869924d'
    inventory = [{**r, 'audio_sha256': digest(data.root / r['path'])} for r in data.rows]
    atomic_json(root / 'full_training_inventory.json', inventory)
    protocol = {'purpose': 'final_refit', 'input_files': len(data.rows), 'original_roles': counts,
                'warmup_epochs': 8, 'finetune_epochs': 17,
                'initialization': 'fresh pinned AudioSet AST; fresh heads, full-data head warmup',
                'checkpoint_selection': 'fixed final epoch; no validation selection',
                'independent_validation_score': None, 'deadline_unix': deadline,
                'device': str(device), 'python': sys.version, 'torch': torch.__version__,
                'training_inventory_sha256': digest(root / 'full_training_inventory.json')}
    atomic_json(root / 'protocol.json', protocol)
    processor = load_processor(args.assets)
    model = ASTMultiTask.from_pretrained(args.assets, 'frozen')
    warmup = train_stage(model, processor, RefitJob(data, root / 'warmup', device, 8, deadline-120))
    model.configure_trainable('last_block')
    finetune = train_stage(model, processor, RefitJob(data, root / 'finetune', device, 17, deadline-120))
    assert all(v == 8 for v in warmup['exposures'].values()) and len(warmup['exposures']) == 1029
    assert all(v == 17 for v in finetune['exposures'].values()) and len(finetune['exposures']) == 1029
    checkpoint = root / 'final_checkpoint'
    export_checkpoint(model, processor, checkpoint)
    del model
    gc.collect()
    if device.type == 'mps':
        torch.mps.empty_cache()
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    archive = build_ast_submission(checkpoint, root / 'DACON_236749_AST_full1029_epoch17.zip')
    unpacked = root / 'offline_replay'
    with zipfile.ZipFile(archive) as packed:
        assert packed.testzip() is None
        assert digest(checkpoint / 'model.safetensors') == __import__('hashlib').sha256(packed.read('model/ast/model.safetensors')).hexdigest()
        packed.extractall(unpacked)
    reference = root / 'reference'
    (reference / 'model').mkdir(parents=True)
    shutil.copytree(checkpoint, reference / 'model/ast')
    audio_dir = reference / 'data/test'
    audio_dir.mkdir(parents=True)
    # These transformed public clips verify deployment, not independent accuracy.
    waveform, rate = sf.read(data.root / data.rows[0]['path'], dtype='float32')
    sf.write(audio_dir / 'mono.wav', waveform, rate)
    sf.write(audio_dir / 'stereo.wav', np.column_stack((waveform, waveform*.7)), rate)
    sf.write(audio_dir / 'stress_60s.wav', np.tile(waveform, int(np.ceil(60*rate/len(waveform))))[:60*rate], rate)
    ids = ['mono', 'stereo', 'stress_60s']
    write_csv(reference / 'data/sample_submission.csv', [{'ID': i, **{c: 0 for c in PREDICTION_COLUMNS}} for i in ids])
    generate_submission(reference)
    shutil.copytree(reference / 'data', unpacked / 'data')
    guard = root / 'network_guard'
    guard.mkdir()
    (guard / 'sitecustomize.py').write_text("import socket\ndef deny(*a,**kw):\n    raise RuntimeError('offline replay forbids network')\nsocket.socket.connect=deny\nsocket.create_connection=deny\n")
    previous = os.environ.get('PYTHONPATH')
    os.environ['PYTHONPATH'] = str(guard)
    try:
        predicted = verify_packaged(unpacked, deadline)
    finally:
        if previous is None:
            os.environ.pop('PYTHONPATH', None)
        else:
            os.environ['PYTHONPATH'] = previous
    import csv
    with (reference / 'output/submission.csv').open() as stream:
        expected = list(csv.DictReader(stream))
    assert [r['ID'] for r in predicted] == ids
    delta = max(abs(float(a[c])-float(b[c])) for a,b in zip(expected,predicted,strict=True) for c in PREDICTION_COLUMNS)
    assert delta <= 1e-5, delta
    atomic_json(root / 'verification.json', {'network_blocked': True, 'max_delta': delta,
                'files': ids, 'inference_device': 'cuda' if torch.cuda.is_available() else 'cpu',
                'L4_full_runtime': 'unverified', 'zip_sha256': digest(archive),
                'zip_bytes': archive.stat().st_size,
                'checkpoint_sha256': digest(checkpoint / 'model.safetensors')})
    atomic_json(root / 'finished.json', {'status': 'ready_not_submitted', 'full_training_files': 1029,
                'warmup_epochs': 8, 'finetune_epochs': 17, 'independent_validation_score': None,
                'elapsed_seconds': time.time()-started})
    print('FULL_REFIT_READY', archive, flush=True)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/dacon_full_refit/train_v2.py
"""Bounded AST training on development data only."""

from __future__ import annotations

import argparse
import csv
import hashlib
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import torch

from graph_features import atomic_json, digest
from graph_scoring import LABEL_COLUMNS, PREDICTION_COLUMNS, official_metrics
from v2_inference import load_audio as _load_audio

SEED = 20260913
RESERVE_SECONDS = 20 * 60
CONFIGS = {
    "AST_FROZEN": {"mode": "frozen", "epochs": 8, "batch_size": 16,
                   "accumulation": 1, "head_lr": 3e-4, "backbone_lr": 0.0},
    "AST_LAST_BLOCK": {"mode": "last_block", "epochs": 6, "batch_size": 8,
                       "accumulation": 2, "head_lr": 1e-4, "backbone_lr": 1e-5},
}


def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False


def _json_hash(value: object) -> str:
    raw = json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=True)
    return hashlib.sha256(raw.encode()).hexdigest()


def read_development_manifest(path: Path, data_root: Path) -> list[dict[str, str]]:
    """Reject locked-test inputs before any audio file is opened."""
    resolved = path.resolve()
    if path.name != "manifest.csv" or path.parent.name != "development" or "locked_test" in resolved.parts:
        raise ValueError("training accepts development manifest only")
    with path.open(encoding="utf-8-sig", newline="") as stream:
        rows = list(csv.DictReader(stream))
    if not rows or len({row["ID"] for row in rows}) != len(rows):
        raise ValueError("development manifest must contain unique rows")
    for row in rows:
        role = row.get("role", row.get("split", ""))
        if role not in {"dev_train", "dev_select"}:
            raise ValueError("training accepts development roles only")
        audio = (data_root / row["path"]).resolve()
        if not audio.is_relative_to(data_root.resolve()):
            raise ValueError("manifest audio path escapes data root")
    return rows


def _train_window(audio: np.ndarray, identifier: str, epoch: int) -> np.ndarray:
    rng = random.Random(f"{SEED}:{epoch}:{identifier}")
    if len(audio) > 160000:
        start = rng.randrange(len(audio) - 160000 + 1)
        audio = audio[start:start + 160000]
    gain = 10 ** (rng.uniform(-6.0, 6.0) / 20)
    return np.clip(audio * gain, -1.0, 1.0).astype(np.float32)


class FrozenEmbeddingCache:
    """Content-address frozen pooler values; provenance drift creates a new key."""

    def __init__(self, directory: Path, model_sha: str, processor_signature: str):
        self.directory = directory
        self.signature = _json_hash({"model_sha256": model_sha,
                                     "processor": processor_signature,
                                     "augmentation": "seed20260913-fixed-crop-gain-v1"})
        self.misses = 0

    def path_for(self, audio_path: Path, identifier: str) -> Path:
        key = hashlib.sha256(f"{digest(audio_path)}:{identifier}:{self.signature}".encode()).hexdigest()
        return self.directory / f"{key}.pt"

    def get(self, audio_path: Path, identifier: str) -> torch.Tensor | None:
        path = self.path_for(audio_path, identifier)
        if not path.exists():
            self.misses += 1
            if self.misses % 25 == 0:
                print(f"FROZEN_CACHE {self.misses} embeddings", flush=True)
            return None
        record = torch.load(path, map_location="cpu", weights_only=True)
        if record["signature"] != self.signature or record["audio_sha256"] != digest(audio_path):
            raise ValueError("frozen embedding cache provenance mismatch")
        return record["embedding"]

    def put(self, audio_path: Path, identifier: str, embedding: torch.Tensor) -> None:
        path = self.path_for(audio_path, identifier)
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_suffix(".tmp")
        torch.save({"signature": self.signature, "audio_sha256": digest(audio_path),
                    "embedding": embedding.detach().cpu()}, temporary)
        temporary.replace(path)


def _targets(rows: list[dict[str, str]], device: torch.device) -> torch.Tensor:
    values = [[float(row[column]) if row.get(column, "") != "" else math.nan
               for column in LABEL_COLUMNS] for row in rows]
    return torch.tensor(values, dtype=torch.float32, device=device)


def _proxy(report: dict) -> float:
    weights = {"file_eer": .45, "voice_eer": .18, "music_eer": .27,
               "voice_presence_auc": .05, "music_presence_auc": .05}
    available = {name: value for name, value in report["metrics"].items() if value is not None}
    total = sum(weights[name] for name in available)
    if not total:
        return float("-inf")
    return sum(weights[name] * ((1 - value) if name.endswith("eer") else value)
               for name, value in available.items()) / total


def _trainable_state(model: torch.nn.Module) -> dict[str, torch.Tensor]:
    names = {name for name, parameter in model.named_parameters() if parameter.requires_grad}
    return {name: value.detach().cpu().clone() for name, value in model.state_dict().items() if name in names}


def _rng_state() -> dict:
    return {"python": random.getstate(), "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}


def _restore_rng(state: dict) -> None:
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if state["cuda"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(state["cuda"])


def save_training_checkpoint(path: Path, model: torch.nn.Module, optimizer: torch.optim.Optimizer,
                             epoch: int, config_hash: str, reports: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save({"epoch": epoch, "config_hash": config_hash, "model": _trainable_state(model),
                "optimizer": optimizer.state_dict(), "rng": _rng_state(), "reports": reports}, temporary)
    temporary.replace(path)


def load_training_checkpoint(path: Path, model: torch.nn.Module, optimizer: torch.optim.Optimizer,
                             config_hash: str) -> tuple[int, list[dict]]:
    state = torch.load(path, map_location="cpu", weights_only=False)
    if state["config_hash"] != config_hash:
        raise ValueError("checkpoint configuration mismatch")
    missing, unexpected = model.load_state_dict(state["model"], strict=False)
    trainable = {name for name, parameter in model.named_parameters() if parameter.requires_grad}
    if unexpected or trainable.intersection(missing):
        raise ValueError("checkpoint trainable parameters do not match model")
    optimizer.load_state_dict(state["optimizer"])
    _restore_rng(state["rng"])
    return int(state["epoch"]), list(state["reports"])


def _optimizer(model: torch.nn.Module, config: dict) -> torch.optim.AdamW:
    head_ids = {id(parameter) for parameter in model.heads.parameters()}
    heads = [parameter for parameter in model.parameters() if parameter.requires_grad and id(parameter) in head_ids]
    backbone = [parameter for parameter in model.parameters() if parameter.requires_grad and id(parameter) not in head_ids]
    groups = [{"params": heads, "lr": config["head_lr"]}]
    if backbone:
        groups.append({"params": backbone, "lr": config["backbone_lr"]})
    return torch.optim.AdamW(groups, weight_decay=.01)


def extend_checkpoint(source: Path, destination: Path, old_config: dict, new_config: dict) -> None:
    """Preserve optimizer/RNG state while explicitly extending only the training budget."""
    allowed = {"epochs", "patience"}
    if {k: v for k, v in old_config.items() if k not in allowed} != {
        k: v for k, v in new_config.items() if k not in allowed
    }:
        raise ValueError("extension may change only epochs and patience")
    if destination.exists() or source.resolve() == destination.resolve():
        raise ValueError("extension requires a new checkpoint destination")
    state = torch.load(source, map_location="cpu", weights_only=False)
    if state["config_hash"] != _json_hash(old_config):
        raise ValueError("source checkpoint configuration mismatch")
    if new_config["epochs"] <= max(old_config["epochs"], state["epoch"]):
        raise ValueError("extension must increase the epoch budget")
    state["config_hash"] = _json_hash(new_config)
    destination.parent.mkdir(parents=True, exist_ok=True)
    torch.save(state, destination)


def _batch_inputs(rows: list[dict[str, str]], root: Path, processor, epoch: int, device: torch.device):
    from ast_multitask import prepare_inputs
    windows = [_train_window(_load_audio(root / row["path"]), row["ID"], epoch) for row in rows]
    return prepare_inputs(processor, tuple(windows)).to(device), _targets(rows, device)


def _frozen_logits(model, processor, rows: list[dict[str, str]], root: Path,
                   cache: FrozenEmbeddingCache, device: torch.device) -> torch.Tensor:
    from ast_multitask import prepare_inputs
    embeddings = []
    for row in rows:
        path = root / row["path"]
        embedding = cache.get(path, row["ID"])
        if embedding is None:
            window = _train_window(_load_audio(path), row["ID"], 0)
            with torch.no_grad():
                embedding = model.encode(prepare_inputs(processor, (window,)).to(device))[0].cpu()
            cache.put(path, row["ID"], embedding)
        embeddings.append(embedding)
    pooled = torch.stack(embeddings).to(device)
    return torch.cat([head(pooled) for head in model.heads], dim=-1)


@torch.no_grad()
def predict_rows(model, processor, rows: list[dict[str, str]], root: Path,
                 device: torch.device,
                 deadline: float | None = None,
                 cache: FrozenEmbeddingCache | None = None) -> list[dict[str, str | float]]:
    from ast_multitask import aggregate_window_logits, prepare_inputs, split_audio
    model.eval()
    predictions = []
    for row in rows:
        if deadline is not None and time.time() >= deadline:
            raise TimeoutError("prediction deadline reached before completing a file")
        path = root / row["path"]
        embeddings = cache.get(path, row["ID"] + "#eval") if cache else None
        if embeddings is None:
            windows = split_audio(_load_audio(path))
            if cache:
                embeddings = model.encode(prepare_inputs(processor, windows).to(device)).cpu()
                cache.put(path, row["ID"] + "#eval", embeddings)
        logits = (torch.cat([head(embeddings.to(device)) for head in model.heads], dim=-1)
                  if cache else model(prepare_inputs(processor, windows).to(device)))
        probabilities = aggregate_window_logits(logits).float().cpu().tolist()
        predictions.append({"ID": row["ID"], **dict(zip(PREDICTION_COLUMNS, probabilities, strict=True))})
    return predictions


def train_candidate(name: str, model, processor, rows: list[dict[str, str]], data_root: Path,
                    output: Path, deadline: float, manifest_hash: str,
                    asset_model_sha: str) -> tuple[object, dict]:
    from ast_multitask import compute_pos_weights, export_checkpoint, masked_multitask_loss
    config = {**CONFIGS[name], "seed": SEED, "manifest_sha256": manifest_hash,
              "asset_model_sha256": asset_model_sha,
              "weight_decay": .01, "gradient_clip": 1.0}
    config_hash = _json_hash(config)
    if time.time() >= deadline - RESERVE_SECONDS:
        raise TimeoutError(f"not enough time to start {name}")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    optimizer = _optimizer(model, config)
    train = [row for row in rows if row.get("role", row.get("split")) == "dev_train"]
    select = [row for row in rows if row.get("role", row.get("split")) == "dev_select"]
    pos_weights = compute_pos_weights(_targets(train, device))
    processor_signature = _json_hash({name: getattr(processor, name) for name in
                                      ("sampling_rate", "num_mel_bins", "max_length", "mean", "std")})
    frozen_cache = FrozenEmbeddingCache(output / name / "pooler_cache", asset_model_sha,
                                        processor_signature) if name == "AST_FROZEN" else None
    checkpoint = output / name / "last.pt"
    start_epoch, reports = (0, [])
    if checkpoint.exists():
        start_epoch, reports = load_training_checkpoint(checkpoint, model, optimizer, config_hash)
    best_proxy = max((report["proxy"] for report in reports), default=float("-inf"))
    best_epoch = next((r["epoch"] for r in reports if r["proxy"] == best_proxy), 0)
    best_state = None
    if start_epoch and (output / name / "best.pt").exists():
        best_state = torch.load(output / name / "best.pt", map_location="cpu", weights_only=True)
    for epoch in range(start_epoch + 1, config["epochs"] + 1):
        if config.get("patience", 0) and epoch - 1 - best_epoch >= config["patience"]:
            break
        if time.time() >= deadline - RESERVE_SECONDS:
            break
        started = time.perf_counter()
        model.train()
        before_epoch = {key: value.clone() for key, value in _trainable_state(model).items()}
        optimizer.zero_grad(set_to_none=True)
        shuffled = list(train)
        random.Random(SEED + epoch).shuffle(shuffled)
        losses = []
        gradient_norms = []
        head_gradient_norms = []
        backbone_gradient_norms = []
        for batch_index in range(0, len(shuffled), config["batch_size"]):
            if time.time() >= deadline - RESERVE_SECONDS:
                raise TimeoutError(f"deadline reached during {name}; incomplete epoch was discarded")
            batch = shuffled[batch_index:batch_index + config["batch_size"]]
            targets = _targets(batch, device)
            amp = torch.autocast("cuda", dtype=torch.bfloat16) if device.type == "cuda" else torch.autocast("cpu", enabled=False)
            with amp:
                logits = (_frozen_logits(model, processor, batch, data_root, frozen_cache, device)
                          if frozen_cache else model(_batch_inputs(batch, data_root, processor, 0, device)[0]))
                loss = masked_multitask_loss(logits, targets, pos_weights) / config["accumulation"]
            loss.backward()
            losses.append(float(loss.detach()) * config["accumulation"])
            step = (batch_index // config["batch_size"] + 1) % config["accumulation"] == 0
            last = batch_index + config["batch_size"] >= len(shuffled)
            if step or last:
                heads = [p for p in model.heads.parameters() if p.requires_grad]
                backbone = [p for parameter_name, p in model.named_parameters()
                            if p.requires_grad and not parameter_name.startswith("heads.")]
                head_gradient_norms.append(float(torch.nn.utils.clip_grad_norm_(heads, float("inf"))))
                if backbone:
                    backbone_gradient_norms.append(float(torch.nn.utils.clip_grad_norm_(backbone, float("inf"))))
                norm = torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
                gradient_norms.append(float(norm))
                if not math.isfinite(gradient_norms[-1]):
                    raise FloatingPointError(f"non-finite gradient in {name}")
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
        if max(head_gradient_norms, default=0.0) <= 0 or (
            name == "AST_LAST_BLOCK" and max(backbone_gradient_norms, default=0.0) <= 0
        ):
            raise FloatingPointError(f"intended trainable path received no gradient in {name}")
        predictions = predict_rows(model, processor, select, data_root, device,
                                   deadline=deadline - RESERVE_SECONDS, cache=frozen_cache)
        metric_report = official_metrics(select, predictions)
        proxy = _proxy(metric_report)
        report = {"epoch": epoch, "mean_loss": float(np.mean(losses)), "proxy": proxy,
                  "metric_report": metric_report, "seconds": time.perf_counter() - started,
                  "max_gradient_norm": max(gradient_norms, default=0.0),
                  "max_head_gradient_norm": max(head_gradient_norms, default=0.0),
                  "max_backbone_gradient_norm": max(backbone_gradient_norms, default=0.0),
                  "trainable_absolute_update": sum(
                      float((value - before_epoch[key]).abs().sum())
                      for key, value in _trainable_state(model).items()),
                  "head_absolute_update": sum(
                      float((value - before_epoch[key]).abs().sum())
                      for key, value in _trainable_state(model).items() if key.startswith("heads.")),
                  "backbone_absolute_update": sum(
                      float((value - before_epoch[key]).abs().sum())
                      for key, value in _trainable_state(model).items() if not key.startswith("heads."))}
        reports.append(report)
        if proxy > best_proxy:
            best_proxy, best_state = proxy, _trainable_state(model)
            best_epoch = epoch
            best_path = output / name / "best.pt"
            best_path.parent.mkdir(parents=True, exist_ok=True)
            temporary = best_path.with_suffix(".tmp")
            torch.save(best_state, temporary)
            temporary.replace(best_path)
        save_training_checkpoint(checkpoint, model, optimizer, epoch, config_hash, reports)
        atomic_json(output / name / f"epoch_{epoch:03d}.json", report)
        print(f"TRAIN_EPOCH {name} {epoch}/{config['epochs']} "
              f"loss={report['mean_loss']:.6f} proxy={proxy:.6f} "
              f"head_grad={report['max_head_gradient_norm']:.6f} "
              f"backbone_grad={report['max_backbone_gradient_norm']:.6f}", flush=True)
    if best_state is None:
        raise TimeoutError(f"{name} completed no epoch")
    model.load_state_dict(best_state, strict=False)
    exported = output / name / "best"
    export_checkpoint(model, processor, exported)
    predictions = predict_rows(model, processor, select, data_root, device,
                               deadline=deadline - RESERVE_SECONDS, cache=frozen_cache)
    changed = max(report.get("trainable_absolute_update", 0.0) for report in reports)
    trainable_names = [name for name, parameter in model.named_parameters() if parameter.requires_grad]
    result = {"candidate": name, "checkpoint": str(exported), "config": config,
              "config_hash": config_hash, "best_proxy": best_proxy, "epochs_completed": len(reports),
              "best_epoch": best_epoch,
              "predictions": predictions, "reports": reports, "trainable_names": trainable_names,
              "trainable_parameters": sum(p.numel() for p in model.parameters() if p.requires_grad),
              "trainable_absolute_update": changed,
              "peak_allocated_gib": torch.cuda.max_memory_allocated() / 1024**3 if device.type == "cuda" else 0.0,
              "peak_reserved_gib": torch.cuda.max_memory_reserved() / 1024**3 if device.type == "cuda" else 0.0}
    prediction_path = output / name / "dev_predictions.csv"
    with prediction_path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=["ID", *PREDICTION_COLUMNS])
        writer.writeheader()
        writer.writerows(predictions)
    atomic_json(output / name / "report.json", result)
    return model, result


def train_candidates(ast_assets: Path, manifest_path: Path, data_root: Path,
                     output: Path, deadline: float) -> dict:
    from ast_multitask import ASTMultiTask, load_processor
    seed_everything()
    rows = read_development_manifest(manifest_path, data_root)
    processor = load_processor(ast_assets)
    asset_model_sha = digest(ast_assets / "model.safetensors")
    results = {}
    frozen, results["AST_FROZEN"] = train_candidate(
        "AST_FROZEN", ASTMultiTask.from_pretrained(ast_assets, "frozen"), processor,
        rows, data_root, output, deadline, digest(manifest_path), asset_model_sha)
    if time.time() < deadline - RESERVE_SECONDS:
        last = ASTMultiTask.from_pretrained(ast_assets, "last_block")
        last.heads.load_state_dict(frozen.heads.state_dict())
        _, results["AST_LAST_BLOCK"] = train_candidate(
            "AST_LAST_BLOCK", last, processor, rows, data_root, output, deadline,
            digest(manifest_path), asset_model_sha)
    else:
        results["AST_LAST_BLOCK"] = {"candidate": "AST_LAST_BLOCK", "status": "skipped_deadline"}
    report = {"candidates": results, "manifest_sha256": digest(manifest_path),
              "finished_unix": time.time()}
    atomic_json(output / "training_report.json", report)
    return report


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--assets", type=Path, required=True)
    parser.add_argument("--manifest", type=Path, required=True)
    parser.add_argument("--data-root", type=Path, required=True)
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--deadline", type=float, required=True)
    args = parser.parse_args()
    train_candidates(args.assets, args.manifest, args.data_root, args.output, args.deadline)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/dacon_full_refit/ast_multitask.py
"""Pinned AST encoder with five independently trainable DACON heads."""

from __future__ import annotations

import json
import math
from pathlib import Path
from typing import Final, Literal, assert_never

import numpy as np
import torch
import torch.nn.functional as functional
from safetensors.torch import load_file, save_file
from torch import nn
from transformers import ASTConfig, ASTFeatureExtractor, ASTModel

PREDICTION_COLUMNS: Final = (
    "FILE_FAKE_PROB",
    "VOICE_FAKE_PROB",
    "MUSIC_FAKE_PROB",
    "VOICE_PRESENT_PROB",
    "MUSIC_PRESENT_PROB",
)
SAMPLE_RATE: Final = 16_000
WINDOW_SAMPLES: Final = 160_000
PROCESSOR_CONTRACT: Final = {
    "sampling_rate": SAMPLE_RATE,
    "num_mel_bins": 128,
    "max_length": 1024,
    "mean": -4.2677393,
    "std": 4.5689974,
}
TrainMode = Literal["frozen", "last_block"]


class ASTContractError(ValueError):
    """An offline AST asset or input violates the pinned model contract."""


def _validate_processor(processor: ASTFeatureExtractor) -> None:
    for name, expected in PROCESSOR_CONTRACT.items():
        actual = getattr(processor, name, None)
        if not isinstance(actual, int | float) or not math.isclose(float(actual), expected):
            raise ASTContractError("AST processor contract mismatch")


def load_processor(local_dir: Path) -> ASTFeatureExtractor:
    """Load the pinned processor locally and fail closed on configuration drift."""
    processor = ASTFeatureExtractor.from_pretrained(local_dir, local_files_only=True)
    _validate_processor(processor)
    return processor


def _mps_block_input_gradient(module: nn.Module, inputs: tuple) -> tuple:
    """Avoid MPS partial LayerNorm backward errors without unfreezing the prefix."""
    hidden = inputs[0]
    if (hidden.device.type == "mps" and torch.is_grad_enabled()
            and module.layernorm_before.weight.requires_grad):
        hidden.requires_grad_(True)
    return inputs


class ASTMultiTask(nn.Module):
    """AST pooler followed by five independent binary heads."""

    def __init__(self, backbone: ASTModel, mode: TrainMode) -> None:
        super().__init__()
        self.backbone = backbone
        self.backbone.encoder.layer[-1].register_forward_pre_hook(_mps_block_input_gradient)
        self.heads = nn.ModuleList(nn.Linear(backbone.config.hidden_size, 1) for _ in PREDICTION_COLUMNS)
        self.mode = mode
        self.configure_trainable(mode)

    @classmethod
    def from_pretrained(cls, local_dir: Path, mode: TrainMode) -> ASTMultiTask:
        """Load only supplied local AST weights; AudioSet classifier logits are unused."""
        load_processor(local_dir)
        backbone = ASTModel.from_pretrained(
            local_dir,
            local_files_only=True,
            attn_implementation="sdpa",
        )
        return cls(backbone, mode)

    def configure_trainable(self, mode: TrainMode) -> None:
        """Expose exactly the parameters owned by the selected training mode."""
        for parameter in self.parameters():
            parameter.requires_grad_(False)
        for parameter in self.heads.parameters():
            parameter.requires_grad_(True)
        match mode:
            case "frozen":
                pass
            case "last_block":
                for parameter in self.backbone.encoder.layer[-1].parameters():
                    parameter.requires_grad_(True)
                for parameter in self.backbone.layernorm.parameters():
                    parameter.requires_grad_(True)
            case unreachable:
                assert_never(unreachable)
        self.mode = mode

    def train(self, mode: bool = True) -> ASTMultiTask:
        """Keep frozen encoder blocks deterministic while training selected layers."""
        super().train(mode)
        if mode:
            self.backbone.eval()
            if self.mode == "last_block":
                self.backbone.encoder.layer[-1].train()
        return self

    def encode(self, input_values: torch.Tensor) -> torch.Tensor:
        """Return the final-normalized CLS/distillation-token pooler output."""
        return self.backbone(input_values=input_values).pooler_output

    def forward(self, input_values: torch.Tensor) -> torch.Tensor:
        pooled = self.encode(input_values)
        return torch.cat([head(pooled) for head in self.heads], dim=-1)


def _eligible_mask(targets: torch.Tensor) -> torch.Tensor:
    if targets.ndim != 2 or targets.shape[1] != len(PREDICTION_COLUMNS):
        raise ASTContractError("AST targets must have five columns")
    known = torch.isfinite(targets)
    if torch.any(known & (targets != 0) & (targets != 1)):
        raise ASTContractError("AST targets must be binary or NaN")
    eligible = known.clone()
    eligible[:, 1] &= known[:, 3] & (targets[:, 3] == 1)
    eligible[:, 2] &= known[:, 4] & (targets[:, 4] == 1)
    return eligible


def compute_pos_weights(targets: torch.Tensor) -> torch.Tensor:
    """Compute bounded training-only positive weights over eligible truth."""
    eligible = _eligible_mask(targets)
    weights = torch.ones(len(PREDICTION_COLUMNS), dtype=targets.dtype, device=targets.device)
    for index in range(len(PREDICTION_COLUMNS)):
        selected = targets[:, index][eligible[:, index]]
        positives = torch.count_nonzero(selected == 1)
        negatives = torch.count_nonzero(selected == 0)
        if positives > 0 and negatives > 0:
            weights[index] = (negatives / positives).clamp(0.25, 4.0)
    return weights


def masked_multitask_loss(
    logits: torch.Tensor,
    targets: torch.Tensor,
    pos_weights: torch.Tensor,
) -> torch.Tensor:
    """Mean five independently normalized BCE heads over known eligible labels."""
    if logits.shape != targets.shape or pos_weights.shape != (len(PREDICTION_COLUMNS),):
        raise ASTContractError("AST loss input contract mismatch")
    eligible = _eligible_mask(targets)
    losses: list[torch.Tensor] = []
    safe_targets = torch.nan_to_num(targets)
    for index in range(len(PREDICTION_COLUMNS)):
        mask = eligible[:, index]
        if torch.any(mask):
            losses.append(functional.binary_cross_entropy_with_logits(
                logits[:, index][mask],
                safe_targets[:, index][mask],
                pos_weight=pos_weights[index],
            ))
    if not losses:
        raise ASTContractError("AST batch has no eligible labels")
    return torch.stack(losses).mean()


def split_audio(
    audio: np.ndarray,
    sample_rate: int = SAMPLE_RATE,
    window_samples: int = WINDOW_SAMPLES,
) -> tuple[np.ndarray, ...]:
    """Cover one mono file with deterministic, non-overlapping windows including its tail."""
    if sample_rate != SAMPLE_RATE or window_samples <= 0 or audio.ndim != 1 or audio.size == 0:
        raise ASTContractError("AST audio input contract mismatch")
    if not np.isfinite(audio).all():
        raise ASTContractError("AST audio contains non-finite samples")
    return tuple(audio[start:start + window_samples] for start in range(0, audio.size, window_samples))


def prepare_inputs(
    processor: ASTFeatureExtractor,
    windows: tuple[np.ndarray, ...],
) -> torch.Tensor:
    """Apply the pinned AST feature extractor to one file's windows."""
    _validate_processor(processor)
    if not windows:
        raise ASTContractError("AST requires at least one audio window")
    batch = processor(list(windows), sampling_rate=SAMPLE_RATE, return_tensors="pt")
    return batch["input_values"]


def aggregate_window_logits(logits: torch.Tensor) -> torch.Tensor:
    """Mean all window probabilities from exactly one file."""
    if logits.ndim != 2 or logits.shape[0] == 0 or logits.shape[1] != len(PREDICTION_COLUMNS):
        raise ASTContractError("AST window logits contract mismatch")
    return torch.sigmoid(logits.float()).mean(dim=0)


def predict_file(
    model: ASTMultiTask,
    processor: ASTFeatureExtractor,
    audio: np.ndarray,
) -> torch.Tensor:
    """Predict one file without reading or retaining another file's state."""
    inputs = prepare_inputs(processor, split_audio(audio)).to(next(model.parameters()).device)
    with torch.inference_mode():
        return aggregate_window_logits(model(inputs)).cpu()


def export_checkpoint(
    model: ASTMultiTask,
    processor: ASTFeatureExtractor,
    directory: Path,
) -> Path:
    """Export a self-contained inference checkpoint without optimizer state."""
    _validate_processor(processor)
    directory.mkdir(parents=True, exist_ok=True)
    model.backbone.config.to_json_file(directory / "config.json")
    processor.save_pretrained(directory)
    metadata = {
        "format_version": 1,
        "mode": model.mode,
        "columns": list(PREDICTION_COLUMNS),
        "hidden_size": model.backbone.config.hidden_size,
        "processor_contract": PROCESSOR_CONTRACT,
    }
    metadata_path = directory / "multitask_config.json"
    temporary_metadata = metadata_path.with_suffix(".json.tmp")
    temporary_metadata.write_text(json.dumps(metadata, sort_keys=True), encoding="utf-8")
    temporary_metadata.replace(metadata_path)
    weights_path = directory / "model.safetensors"
    temporary_weights = weights_path.with_suffix(".safetensors.tmp")
    save_file(
        {name: value.detach().cpu().contiguous() for name, value in model.state_dict().items()},
        temporary_weights,
    )
    temporary_weights.replace(weights_path)
    return directory


def load_checkpoint(
    directory: Path,
    mode: TrainMode,
) -> tuple[ASTMultiTask, ASTFeatureExtractor]:
    """Load a self-contained checkpoint and reject processor/head contract drift."""
    metadata_path = directory / "multitask_config.json"
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    expected = {
        "format_version": 1,
        "mode": mode,
        "columns": list(PREDICTION_COLUMNS),
    }
    if any(metadata.get(name) != value for name, value in expected.items()):
        raise ASTContractError("AST checkpoint contract mismatch")
    processor = load_processor(directory)
    config = ASTConfig.from_json_file(directory / "config.json")
    if metadata.get("hidden_size") != config.hidden_size:
        raise ASTContractError("AST checkpoint contract mismatch")
    model = ASTMultiTask(ASTModel(config), mode)
    model.load_state_dict(load_file(directory / "model.safetensors"), strict=True)
    return model, processor


In [ ]:
%%writefile /content/dacon_full_refit/v2_inference.py
"""Offline, file-local inference for the self-contained v2 AST submission."""

from __future__ import annotations

import csv
import json
import math
import os
from pathlib import Path
from typing import Final

import numpy as np
import soundfile as soundfile
import torch
from scipy.signal import resample_poly
from transformers import ASTFeatureExtractor

from ast_multitask import PREDICTION_COLUMNS, SAMPLE_RATE, ASTMultiTask, load_checkpoint, predict_file

SUBMISSION_COLUMNS: Final = ("ID", *PREDICTION_COLUMNS)
SUPPORTED_AUDIO_SUFFIXES: Final = frozenset({".wav", ".flac", ".mp3"})


class InferenceContractError(ValueError):
    """The evaluator input or local inference asset violates the submission contract."""


def load_audio(path: Path) -> np.ndarray:
    """Decode one evaluator file to finite mono 16 kHz float32 PCM without a network codec."""
    try:
        waveform, sample_rate = soundfile.read(path, dtype="float32", always_2d=True)
    except (OSError, RuntimeError) as error:
        raise InferenceContractError(f"Cannot decode audio file: {path.name}") from error
    if waveform.size == 0 or sample_rate <= 0 or not np.isfinite(waveform).all():
        raise InferenceContractError(f"Invalid audio file: {path.name}")
    mono = waveform.mean(axis=1, dtype=np.float32)
    if sample_rate != SAMPLE_RATE:
        divisor = math.gcd(sample_rate, SAMPLE_RATE)
        mono = resample_poly(mono, SAMPLE_RATE // divisor, sample_rate // divisor).astype(np.float32)
    if mono.size == 0 or not np.isfinite(mono).all():
        raise InferenceContractError(f"Invalid resampled audio file: {path.name}")
    return mono


def _sample_ids(path: Path) -> list[str]:
    """Parse the evaluator-owned ID order once at the CSV trust boundary."""
    try:
        with path.open(encoding="utf-8-sig", newline="") as stream:
            reader = csv.DictReader(stream)
            if reader.fieldnames != list(SUBMISSION_COLUMNS):
                raise InferenceContractError("Sample submission columns differ from the official schema")
            identifiers = [row.get("ID", "") for row in reader]
    except OSError as error:
        raise InferenceContractError("Cannot read sample submission") from error
    if not identifiers or any(not identifier for identifier in identifiers) or len(identifiers) != len(set(identifiers)):
        raise InferenceContractError("Sample submission IDs must be non-empty and unique")
    return identifiers


def ordered_audio_files(test_dir: Path, identifiers: list[str]) -> list[Path]:
    """Map each requested ID to exactly one supported evaluator file."""
    files = [path for path in test_dir.rglob("*") if path.is_file() and path.suffix.lower() in SUPPORTED_AUDIO_SUFFIXES]
    by_id = {path.stem: path for path in files}
    if len(by_id) != len(files):
        raise InferenceContractError("Evaluator audio IDs are not unique")
    missing = set(identifiers) - set(by_id)
    extra = set(by_id) - set(identifiers)
    if missing or extra:
        raise InferenceContractError(f"Evaluator audio IDs mismatch: missing={sorted(missing)[:3]}, extra={sorted(extra)[:3]}")
    return [by_id[identifier] for identifier in identifiers]


def load_packaged_model(root: Path) -> tuple[ASTMultiTask, ASTFeatureExtractor]:
    """Load one local checkpoint, selecting only a declared training mode."""
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    checkpoint = root / "model" / "ast"
    try:
        metadata = json.loads((checkpoint / "multitask_config.json").read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError) as error:
        raise InferenceContractError("Cannot read local AST checkpoint metadata") from error
    mode = metadata.get("mode")
    match mode:
        case "frozen" | "last_block":
            model, processor = load_checkpoint(checkpoint, mode)
        case _:
            raise InferenceContractError("Unsupported local AST checkpoint mode")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return model.eval().to(device), processor


def predict_audio_file(model: ASTMultiTask, processor: ASTFeatureExtractor, path: Path) -> list[float]:
    """Predict one file, rejecting malformed output before it can reach the CSV."""
    probabilities = predict_file(model, processor, load_audio(path)).tolist()
    if len(probabilities) != len(PREDICTION_COLUMNS) or any(
        not math.isfinite(value) or value < 0.0 or value > 1.0 for value in probabilities
    ):
        raise InferenceContractError(f"Invalid probability output for: {path.name}")
    return probabilities


def generate_submission(root: Path) -> Path:
    """Run file-local inference and atomically write evaluator-ordered submission.csv."""
    identifiers = _sample_ids(root / "data" / "sample_submission.csv")
    paths = ordered_audio_files(root / "data" / "test", identifiers)
    model, processor = load_packaged_model(root)
    rows = []
    for index, path in enumerate(paths, 1):
        probabilities = predict_audio_file(model, processor, path)
        rows.append(dict(zip(SUBMISSION_COLUMNS, [path.stem, *probabilities], strict=True)))
        print(f"PREDICT {index}/{len(paths)} {path.stem}", flush=True)
    destination = root / "output" / "submission.csv"
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(".csv.tmp")
    with temporary.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=SUBMISSION_COLUMNS)
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(destination)
    return destination


def main() -> None:
    """Run against an evaluator-layout directory passed as the working directory."""
    os.environ.setdefault("HF_HUB_OFFLINE", "1")
    os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
    generate_submission(Path.cwd())


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/dacon_full_refit/graph_features.py
"""Shared, file-independent model evidence; baseline preprocessing is retained."""

import gc
import hashlib
import importlib.util
import importlib.metadata
import json
import time
from pathlib import Path

import numpy as np
import torch

from graph_fusion import Feature


def atomic_json(path: Path, value) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    temporary.replace(path)


def digest(path: Path) -> str:
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


class Extractor:
    """Own the three frozen models and accumulate timing for one experiment."""

    def __init__(self, baseline_script: Path, include_raw: bool = True):
        spec = importlib.util.spec_from_file_location('official_baseline', baseline_script)
        if spec is None or spec.loader is None:
            raise ImportError(f'Cannot import baseline at {baseline_script}')
        self.base = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(self.base)
        self.device = self.base.select_device('cuda')
        self.include_raw = include_raw
        torch.manual_seed(20260913)
        np.random.seed(20260913)
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        self.panns, self.vi, self.mi = self.base.load_panns_model(self.device)
        self.df, self.fake_index = self.base.load_df_arena_model(self.device)
        self.demucs = self.base.load_htdemucs_model().to(self.device)
        self.signature = hashlib.sha256((
            digest(Path(__file__)) + digest(baseline_script)
            + (self.base.MODEL_DIR / 'SHA256SUMS.txt').read_text()
            + '|'.join(importlib.metadata.version(k) for k in
                ('torch', 'torchaudio', 'transformers', 'librosa', 'demucs', 'panns-inference'))
            + f'fp32-serial-shifts0-overlap0.25-raw{include_raw}'
        ).encode()).hexdigest()

    def fake_windows(self, audio: np.ndarray) -> list[float]:
        starts = self.base.get_segment_starts(audio.size)
        if self.base.calculate_rms(audio) < self.base.SILENCE_RMS:
            return [0.0] * len(starts)
        result = []
        for start in starts:
            segment = torch.from_numpy(self.base.extract_segment(audio, start)).to(self.device)
            with torch.inference_mode():
                logits = self.df(input_values=segment)['logits']
                probability = torch.softmax(logits.float(), dim=-1)[0, self.fake_index]
            result.append(float(probability))
        return result

    def extract(self, audio_path: Path) -> Feature:
        """No other file or accumulated prediction enters this computation."""
        audio = self.base.load_audio(audio_path)
        predictions, _ = self.panns.inference(self.base.make_panns_segments(audio))
        presence = np.stack((predictions[:, self.vi].max(1), predictions[:, self.mi].max(1)), 1)
        voice, music = self.base.separate_voice_and_music(audio_path, self.demucs, self.device)
        feature = Feature(
            ID=audio_path.stem, starts=self.base.get_segment_starts(audio.size),
            presence=presence.astype(float).tolist(),
            voice_fake=self.fake_windows(voice), music_fake=self.fake_windows(music),
            raw_fake=self.fake_windows(audio) if self.include_raw else [0.0] * len(presence),
            stem_rms=[self.base.calculate_rms(voice), self.base.calculate_rms(music)],
            duration=float(audio.size / self.base.AUDIO_SAMPLE_RATE),
        )
        lengths = {len(feature[name]) for name in ('starts', 'presence', 'voice_fake', 'music_fake', 'raw_fake')}
        if len(lengths) != 1:
            raise ValueError(f'Window alignment mismatch: {audio_path}, {lengths}')
        return feature

    def close(self) -> None:
        del self.panns, self.df, self.demucs
        gc.collect()
        torch.cuda.empty_cache()


def extract_cached(extractor: Extractor, paths: list[Path], cache_dir: Path, deadline: float) -> list[Feature]:
    """Only complete atomic records can resume; content and code changes invalidate keys."""
    features = []
    for path in paths:
        key = hashlib.sha256((digest(path) + extractor.signature).encode()).hexdigest()
        cache_path = cache_dir / f'{key}.json'
        if cache_path.is_file():
            record = json.loads(cache_path.read_text())
            if record['signature'] != extractor.signature or record['input_sha256'] != digest(path):
                raise ValueError(f'Cache provenance mismatch: {cache_path}')
            feature = record['feature']
        else:
            if time.time() >= deadline:
                break
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()
            start = time.perf_counter()
            feature = extractor.extract(path)
            torch.cuda.synchronize()
            atomic_json(cache_path, {
                'feature': feature, 'input_sha256': digest(path), 'signature': extractor.signature,
                'seconds': time.perf_counter() - start,
                'max_allocated_gib': torch.cuda.max_memory_allocated() / 1024**3,
                'max_reserved_gib': torch.cuda.max_memory_reserved() / 1024**3,
            })
        features.append({**feature, 'ID': path.stem})
        print(f'FEATURE {len(features)}/{len(paths)} {path.stem}', flush=True)
    return features


In [ ]:
%%writefile /content/dacon_full_refit/graph_fusion.py
"""Five file-independent fusion graphs over one shared audio-inference cache."""

from collections.abc import Sequence
from dataclasses import dataclass
from typing import Final, Literal, TypedDict

import numpy as np
from numpy.typing import NDArray


class Feature(TypedDict):
    ID: str
    starts: list[int]
    presence: list[list[float]]
    voice_fake: list[float]
    music_fake: list[float]
    raw_fake: list[float]
    stem_rms: list[float]
    duration: float


class Label(TypedDict):
    ID: str
    group_id: str
    split: Literal["dev", "holdout"]
    FILE_FAKE: int | str | None
    VOICE_FAKE: int | str | None
    MUSIC_FAKE: int | str | None
    VOICE_PRESENT: int | str | None
    MUSIC_PRESENT: int | str | None


class Prediction(TypedDict):
    ID: str
    FILE_FAKE_PROB: float
    VOICE_FAKE_PROB: float
    MUSIC_FAKE_PROB: float
    VOICE_PRESENT_PROB: float
    MUSIC_PRESENT_PROB: float


class Head(TypedDict, total=False):
    status: str
    reason: str
    n: int
    train_ids: list[str]
    coef: list[float]
    intercept: float
    mean: list[float]
    scale: list[float]


class Fusion(TypedDict):
    version: int
    feature_names: list[str]
    train_ids: list[str]
    train_groups: list[str]
    heads: dict[str, Head]


@dataclass(frozen=True, slots=True)
class CacheError(ValueError):
    detail: str

    def __str__(self) -> str:
        return self.detail


FEATURE_NAMES: Final = ["voice_max", "voice_mean", "music_max", "music_mean",
    "raw_max", "raw_mean", "voice_presence_max", "music_presence_max",
    "voice_raw_disagreement", "music_raw_disagreement", "log_windows"]
HEADS: Final = ("FILE_FAKE_PROB", "VOICE_FAKE_PROB", "MUSIC_FAKE_PROB")
FloatArray = NDArray[np.float64]
GraphPredictions = dict[str, Prediction]


def _arrays(feature: Feature) -> tuple[FloatArray, FloatArray]:
    """Reject corrupted cache probabilities and misaligned window evidence."""
    starts = np.asarray(feature["starts"])
    count = len(starts)
    if not isinstance(feature["ID"], str) or not feature["ID"] or starts.ndim != 1 or count == 0:
        raise CacheError("A nonempty ID and nonempty window starts are required")
    if not np.issubdtype(starts.dtype, np.integer) or np.any(starts < 0) or np.any(np.diff(starts) <= 0):
        raise CacheError("Window starts must be strictly increasing nonnegative integers")
    presence = np.asarray(feature["presence"], dtype=np.float64)
    fake = np.asarray([feature[key] for key in ("voice_fake", "music_fake", "raw_fake")], dtype=np.float64)
    if presence.shape != (count, 2) or fake.shape != (3, count):
        raise CacheError("Presence and fake probabilities must align with every window")
    if any(np.any(~np.isfinite(values) | (values < 0) | (values > 1)) for values in (presence, fake)):
        raise CacheError("Cached probabilities must be finite and inside [0, 1]")
    rms = np.asarray(feature["stem_rms"], dtype=np.float64)
    if rms.shape != (2,) or np.any(~np.isfinite(rms) | (rms < 0)):
        raise CacheError("Stem RMS values must be finite and nonnegative")
    if not np.isfinite(feature["duration"]) or feature["duration"] <= 0:
        raise CacheError("Audio duration must be finite and positive")
    return presence, fake


def _vector(presence: FloatArray, fake: FloatArray) -> FloatArray:
    maxima = fake.max(axis=1)
    return np.concatenate((np.column_stack((maxima, fake.mean(axis=1))).ravel(),
        presence.max(axis=0), np.abs(maxima[:2] - maxima[2]), [np.log1p(fake.shape[1])]))


def feature_vector(feature: Feature) -> FloatArray:
    """Return the fixed, file-local 11-feature input used by the learned heads."""
    return _vector(*_arrays(feature))


def _row(identifier: str, values: Sequence[float]) -> Prediction:
    file_fake, voice_fake, music_fake, voice_present, music_present = values
    return {"ID": identifier, "FILE_FAKE_PROB": round(float(file_fake), 10),
        "VOICE_FAKE_PROB": round(float(voice_fake), 10),
        "MUSIC_FAKE_PROB": round(float(music_fake), 10),
        "VOICE_PRESENT_PROB": round(float(voice_present), 10),
        "MUSIC_PRESENT_PROB": round(float(music_present), 10)}


def graph_predictions(feature: Feature, fusion: Fusion | None = None) -> GraphPredictions:
    """Apply all five frozen graphs; G4 falls back per head until fitted."""
    presence, fake = _arrays(feature)
    vp, mp = presence.max(axis=0)
    vf, mf, raw = fake.max(axis=1)
    baseline = _row(feature["ID"], [max(vf * vp, mf * mp), vf, mf, vp, mp])
    top_vf, top_mf = np.sort(fake[:2], axis=1)[:, -min(2, fake.shape[1]):].mean(axis=1)
    aligned = 1.0 - (1.0 - presence[:, 0] * fake[0]) * (1.0 - presence[:, 1] * fake[1])
    hybrid_vf, hybrid_mf = 0.75 * fake[:2].max(axis=1) + 0.25 * raw
    rows = {"G0": baseline,
        "G1": _row(feature["ID"], [max(top_vf * vp, top_mf * mp), top_vf, top_mf, vp, mp]),
        "G2": _row(feature["ID"], [aligned.max(), vf, mf, vp, mp]),
        "G3": _row(feature["ID"], [max(raw, hybrid_vf * vp, hybrid_mf * mp), hybrid_vf, hybrid_mf, vp, mp]),
        "G4": baseline.copy()}
    if fusion is not None:
        if fusion["version"] != 1 or fusion["feature_names"] != FEATURE_NAMES:
            raise CacheError("Fusion model version or feature order is incompatible")
        vector = _vector(presence, fake)
        for name in HEADS:
            head = fusion["heads"][name]
            if head["status"] == "trained":
                coef, mean, scale = (np.asarray(head[key], dtype=np.float64) for key in ("coef", "mean", "scale"))
                if any(a.shape != vector.shape or np.any(~np.isfinite(a)) for a in (coef, mean, scale)) or np.any(scale <= 0) or not np.isfinite(head["intercept"]):
                    raise CacheError("Fusion parameters must be finite with positive scales")
                logit = float(((vector - mean) / scale) @ coef + head["intercept"])
                probability = np.exp(-np.logaddexp(0.0, -logit))
                rows["G4"][name] = round(float(probability), 10)
            elif head["status"] != "fallback":
                raise CacheError("Unknown fusion head status")
    return rows


def _binary(value: int | str | None) -> int | None:
    if value is None or value == "":
        return None
    if value not in (0, 1, "0", "1"):
        raise CacheError("Ground-truth labels must be binary or missing")
    return int(value)


def fit_fusion(features: Sequence[Feature], labels: Sequence[Label]) -> Fusion:
    """Fit C=0.1 logistic heads on eligible dev rows, never held-out sources."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler

    by_id = {f["ID"]: f for f in features}
    if len(by_id) != len(features) or len({row["ID"] for row in labels}) != len(labels):
        raise CacheError("Feature and label IDs must each be unique")
    groups: dict[str, set[str]] = {"dev": set(), "holdout": set()}
    for row in labels:
        if row["split"] not in groups or not isinstance(row["group_id"], str) or not row["group_id"]:
            raise CacheError("Labels require a dev/holdout split and nonempty group_id")
        source_groups = row["group_id"].split("|")
        if not all(source_groups):
            raise CacheError("Mixtures require nonempty constituent source groups")
        groups[row["split"]].update(source_groups)
        for key in ("FILE_FAKE", "VOICE_FAKE", "MUSIC_FAKE", "VOICE_PRESENT", "MUSIC_PRESENT"):
            _binary(row.get(key))
    if groups["dev"] & groups["holdout"]:
        raise CacheError("Source group overlap between dev and holdout")
    development = sorted((row for row in labels if row["split"] == "dev" and row["ID"] in by_id), key=lambda row: row["ID"])
    vectors = {row["ID"]: feature_vector(by_id[row["ID"]]) for row in development}
    result: Fusion = {"version": 1, "feature_names": FEATURE_NAMES.copy(),
        "train_ids": [row["ID"] for row in development],
        "train_groups": sorted({group for row in development for group in row["group_id"].split("|")}), "heads": {}}
    for name, mask in zip(HEADS, (None, "VOICE_PRESENT", "MUSIC_PRESENT")):
        target = name.removesuffix("_PROB")
        eligible = [row for row in development if _binary(row.get(target)) is not None and (mask is None or _binary(row.get(mask)) == 1)]
        y = np.asarray([_binary(row[target]) for row in eligible], dtype=np.int64)
        head: Head = {"status": "fallback", "reason": "fewer_than_two_classes",
            "n": len(eligible), "train_ids": [row["ID"] for row in eligible]}
        if np.unique(y).size == 2:
            x = np.stack([vectors[row["ID"]] for row in eligible])
            scaler = StandardScaler().fit(x)
            model = LogisticRegression(C=0.1, max_iter=1000, random_state=42).fit(scaler.transform(x), y)
            if np.any(model.n_iter_ >= model.max_iter):
                raise CacheError(f"Fusion head {name} failed to converge")
            head.update(status="trained", reason="", coef=model.coef_[0].tolist(),
                intercept=float(model.intercept_[0]), mean=scaler.mean_.tolist(), scale=scaler.scale_.tolist())
        result["heads"][name] = head
    return result


In [ ]:
%%writefile /content/dacon_full_refit/graph_scoring.py
"""Submission validation and DACON 236749's discrete ROC scoring rule.

Labels are 0/1/None under the official prediction names without ``_PROB``.
Partial submetrics expose coverage; an official total requires complete labels.
"""

from collections.abc import Mapping, Sequence
from itertools import groupby
from math import isfinite
from typing import Final, TypedDict

PREDICTION_COLUMNS: Final = (
    "FILE_FAKE_PROB", "VOICE_FAKE_PROB", "MUSIC_FAKE_PROB",
    "VOICE_PRESENT_PROB", "MUSIC_PRESENT_PROB",
)
SUBMISSION_COLUMNS: Final = ("ID", *PREDICTION_COLUMNS)
LABEL_COLUMNS: Final = tuple(name.removesuffix("_PROB") for name in PREDICTION_COLUMNS)
Row = Mapping[str, str | float | int | None]


class PredictionRow(TypedDict):
    ID: str
    FILE_FAKE_PROB: float
    VOICE_FAKE_PROB: float
    MUSIC_FAKE_PROB: float
    VOICE_PRESENT_PROB: float
    MUSIC_PRESENT_PROB: float


class MetricsReport(TypedDict):
    score: float | None
    metrics: dict[str, float | None]
    invalid: dict[str, str]
    partial: dict[str, str]
    counts: dict[str, int]
    unknown_counts: dict[str, int]
    coverage: dict[str, float]


def _identifier(row: Row) -> str:
    identifier = row.get("ID")
    if not isinstance(identifier, str) or not identifier or identifier != identifier.strip():
        message = f"ID must be a non-empty, unpadded string: {identifier!r}"
        raise ValueError(message)
    return identifier


def validate_predictions(rows: Sequence[Row], expected_ids: Sequence[str]) -> list[PredictionRow]:
    """Parse probabilities and enforce exact columns, IDs, and submission order."""
    if not rows or len(set(expected_ids)) != len(expected_ids):
        message = "Predictions must be non-empty and expected IDs must be unique"
        raise ValueError(message)
    identifiers = [_identifier(row) for row in rows]
    if len(set(identifiers)) != len(identifiers) or identifiers != list(expected_ids):
        message = "Prediction IDs must match the expected unique IDs in order"
        raise ValueError(message)
    parsed = []
    for row, identifier in zip(rows, identifiers, strict=True):
        if set(row) != set(SUBMISSION_COLUMNS):
            message = f"Prediction {identifier} must have exactly the six submission columns"
            raise ValueError(message)
        probabilities = []
        for column in PREDICTION_COLUMNS:
            try:
                probability = float(row[column])
            except (TypeError, ValueError) as error:
                message = f"Invalid probability for {identifier}/{column}"
                raise ValueError(message) from error
            if not isfinite(probability) or not 0 <= probability <= 1:
                message = f"Probability outside finite [0, 1] for {identifier}/{column}"
                raise ValueError(message)
            probabilities.append(probability)
        parsed.append(PredictionRow(
            ID=identifier, FILE_FAKE_PROB=probabilities[0], VOICE_FAKE_PROB=probabilities[1],
            MUSIC_FAKE_PROB=probabilities[2], VOICE_PRESENT_PROB=probabilities[3],
            MUSIC_PRESENT_PROB=probabilities[4],
        ))
    return parsed


def _label(row: Row, column: str) -> int | None:
    value = row.get(column)
    if value is None or value == "":
        return None
    if value not in (0, 1, "0", "1"):
        message = f"Label must be 0, 1, or unknown for {_identifier(row)}/{column}"
        raise ValueError(message)
    return int(value)


def _roc_metrics(targets: Sequence[int], scores: Sequence[float]) -> tuple[float, float]:
    """Match sklearn's full tied-score ROC and the published nearest-point EER."""
    positives = sum(targets)
    negatives = len(targets) - positives
    points = [(0.0, 0.0)]
    true_positives = false_positives = 0
    ranked = sorted(zip(scores, targets, strict=True), reverse=True)
    for _, tied in groupby(ranked, key=lambda pair: pair[0]):
        for _, target in tied:
            true_positives += target
            false_positives += 1 - target
        points.append((false_positives / negatives, true_positives / positives))
    fpr, tpr = min(points, key=lambda point: abs(point[0] - (1 - point[1])))
    fnr = 1 - tpr
    eer = (fpr + fnr) / 2
    auc = sum(
        (right[0] - left[0]) * (right[1] + left[1]) / 2
        for left, right in zip(points, points[1:])
    )
    return eer, auc


def official_metrics(labels: Sequence[Row], predictions: Sequence[Row]) -> MetricsReport:
    """Report known-subset metrics, but require complete truth for an official total.

    Coverage is known eligible labels / (known eligible + unknown relevant labels).
    Unknown presence is relevant because it leaves component eligibility unknown.
    """
    parsed = validate_predictions(predictions, [_identifier(row) for row in labels])
    truth = [{column: _label(row, column) for column in LABEL_COLUMNS} for row in labels]
    report = MetricsReport(
        score=None, metrics={}, invalid={}, partial={}, counts={}, unknown_counts={}, coverage={},
    )
    specifications = (
        ("file_eer", "FILE_FAKE", "FILE_FAKE_PROB", None),
        ("voice_eer", "VOICE_FAKE", "VOICE_FAKE_PROB", "VOICE_PRESENT"),
        ("music_eer", "MUSIC_FAKE", "MUSIC_FAKE_PROB", "MUSIC_PRESENT"),
        ("voice_presence_auc", "VOICE_PRESENT", "VOICE_PRESENT_PROB", None),
        ("music_presence_auc", "MUSIC_PRESENT", "MUSIC_PRESENT_PROB", None),
    )
    for name, target_column, score_column, mask_column in specifications:
        targets = []
        scores = []
        missing = 0
        for row, prediction in zip(truth, parsed, strict=True):
            if mask_column is not None:
                if row[mask_column] is None:
                    missing += 1
                    continue
                if row[mask_column] == 0:
                    continue
            target = row[target_column]
            if target is None:
                missing += 1
                continue
            targets.append(target)
            scores.append(float(prediction[score_column]))
        report["counts"][name] = len(targets)
        report["unknown_counts"][name] = missing
        denominator = len(targets) + missing
        report["coverage"][name] = len(targets) / denominator if denominator else 0.0
        report["metrics"][name] = None
        if missing:
            report["partial"][name] = f"known subset only; unknown relevant labels: {missing}"
        if len(set(targets)) != 2:
            report["invalid"][name] = "both positive and negative labels are required"
            continue
        eer, auc = _roc_metrics(targets, scores)
        report["metrics"][name] = auc if name.endswith("auc") else eer
    if not report["invalid"] and not report["partial"]:
        values = [value for value in report["metrics"].values() if value is not None]
        file_eer, voice_eer, music_eer, voice_auc, music_auc = values
        report["score"] = (
            0.45 * (1 - file_eer) + 0.18 * (1 - voice_eer) + 0.27 * (1 - music_eer)
            + 0.05 * voice_auc + 0.05 * music_auc
        )
    return report


In [ ]:
%%writefile /content/dacon_full_refit/run_experiment.py
"""A bounded pilot; every GPU result is checkpointed to the mounted Drive."""

import argparse
import csv
import importlib.metadata
import json
import random
import sys
import time
from pathlib import Path

import numpy as np
import soundfile as sf
import torch

from graph_features import Extractor, atomic_json, extract_cached
from graph_fusion import fit_fusion, graph_predictions
from graph_scoring import official_metrics, validate_predictions


def read_csv(path: Path) -> list:
    with path.open(encoding='utf-8-sig', newline='') as stream:
        return list(csv.DictReader(stream))


def write_csv(path: Path, rows: list) -> None:
    with path.open('w', encoding='utf-8', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def reference_parity(extractor: Extractor, path: Path, feature) -> dict:
    """Compare the shared graph with untouched official functions on real audio."""
    base = extractor.base
    audio = base.load_audio(path)
    vp, mp = base.predict_presence(extractor.panns, extractor.vi, extractor.mi, audio)
    extractor.demucs.cpu()
    voice, music = base.separate_voice_and_music(path, extractor.demucs, extractor.device)
    extractor.demucs.to(extractor.device)
    vf = base.predict_fake(extractor.df, extractor.fake_index, voice, extractor.device)
    mf = base.predict_fake(extractor.df, extractor.fake_index, music, extractor.device)
    expected = np.array([base.combine_file_fake_score(vf, mf, vp, mp), vf, mf, vp, mp])
    actual = np.array(list(graph_predictions(feature)['G0'].values())[1:])
    delta = float(np.max(np.abs(expected - actual)))
    if not np.allclose(expected, actual, rtol=1e-5, atol=1e-6):
        raise AssertionError(f'G0 differs from official inference by {delta}: {path}')
    return {'ID': path.stem, 'max_probability_delta': delta, 'reference': expected.tolist()}


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument('--baseline', type=Path, required=True)
    parser.add_argument('--validation', type=Path, required=True)
    parser.add_argument('--output', type=Path, required=True)
    parser.add_argument('--deadline', type=float, required=True)
    args = parser.parse_args()
    out = args.output
    out.mkdir(parents=True, exist_ok=True)
    start = time.time()
    if start >= args.deadline:
        raise TimeoutError('The existing GPU budget has expired; no new run was started')
    labels = read_csv(args.validation / 'manifest.csv')
    if len({r['ID'] for r in labels}) != len(labels):
        raise ValueError('Duplicate validation IDs')
    paths = {r['ID']: args.validation / r['path'] for r in labels}
    if any(p.stem != key or not p.is_file() for key, p in paths.items()):
        raise ValueError('Manifest ID/path mismatch or missing audio')
    atomic_json(out / 'environment.json', {
        'python': sys.version, 'gpu': torch.cuda.get_device_name(),
        'gpu_total_gib': torch.cuda.get_device_properties(0).total_memory / 1024**3,
        'started_unix': start, 'deadline_unix': args.deadline,
        'packages': {k: importlib.metadata.version(k) for k in
            ('torch', 'torchaudio', 'transformers', 'numpy', 'scipy', 'scikit-learn', 'demucs', 'librosa')},
    })
    extractor = Extractor(args.baseline / 'script.py')
    try:
        dummy = sorted((args.baseline / 'data' / 'test').glob('*.wav'))
        dummy_features = extract_cached(extractor, dummy, out / 'cache', args.deadline)
        atomic_json(out / 'dummy_features.json', dummy_features)
        ordered = list(paths.values())
        random.Random(20260913).shuffle(ordered)
        first = extract_cached(extractor, ordered[:1], out / 'cache', args.deadline)
        if not first:
            raise TimeoutError('No validation feature completed within the budget')
        atomic_json(out / 'reference_parity.json', reference_parity(extractor, ordered[0], first[0]))
        stress_path = args.baseline / 'stress_60s.wav'
        signal = extractor.base.load_audio(ordered[0])
        sf.write(stress_path, np.tile(signal, 960000 // len(signal) + 1)[:960000], 16000)
        try:
            stress = extract_cached(extractor, [stress_path], out / 'stress_cache', args.deadline)
            atomic_json(out / 'stress60.json', {'completed': bool(stress),
                'purpose': '60-second runtime and memory only; repeated source is not validation evidence'})
        except torch.cuda.OutOfMemoryError as error:
            atomic_json(out / 'stress60.json', {'completed': False, 'error': str(error),
                'submission_supported': False})
            torch.cuda.empty_cache()
        features = extract_cached(extractor, ordered, out / 'cache', args.deadline)
    finally:
        extractor.close()
    atomic_json(out / 'features.json', features)
    available = {f['ID'] for f in features}
    completed_labels = [r for r in labels if r['ID'] in available]
    fusion = fit_fusion(features, labels)
    atomic_json(out / 'fusion.json', fusion)
    predictions = {name: [] for name in ('G0', 'G1', 'G2', 'G3', 'G4')}
    for feature in features:
        for name, row in graph_predictions(feature, fusion).items():
            predictions[name].append(row)
    report = {'complete': len(features) == len(labels), 'completed': len(features),
        'planned': len(labels), 'elapsed_seconds': time.time() - start,
        'scope': 'external_source_group_holdout_not_DACON_leaderboard', 'graphs': {}}
    heldout = [r for r in completed_labels if r['split'] == 'holdout']
    heldout_ids = {r['ID'] for r in heldout}
    heldout_groups = {g for r in heldout for g in r['group_id'].split('|')}
    report['holdout'] = {'clips': len(heldout), 'atomic_source_groups': len(heldout_groups),
        'limitation': 'Transforms and mixtures are correlated; source groups are the independent units'}
    weights = {'file_eer': .45, 'voice_eer': .18, 'music_eer': .27,
        'voice_presence_auc': .05, 'music_presence_auc': .05}
    for name, rows in predictions.items():
        validate_predictions(rows, [f['ID'] for f in features])
        write_csv(out / f'{name}_predictions.csv', rows)
        by_id = {r['ID']: r for r in rows if r['ID'] in heldout_ids}
        selected = [by_id[r['ID']] for r in heldout]
        result = official_metrics(heldout, selected) if heldout else {
            'score': None, 'metrics': {k: None for k in weights},
            'invalid': {k: 'no completed holdout examples' for k in weights},
            'partial': {}, 'counts': {k: 0 for k in weights},
            'unknown_counts': {k: 0 for k in weights}, 'coverage': {k: 0.0 for k in weights}}
        # ponytail: partial-label proxy only selects this pilot; it is never an official score.
        result['selection_proxy'] = sum(weight * ((1 - result['metrics'][metric]) if metric.endswith('eer')
            else result['metrics'][metric]) for metric, weight in weights.items()
            if result['metrics'][metric] is not None)
        report['graphs'][name] = result
        print(name, json.dumps(result, ensure_ascii=False), flush=True)
    ranked = sorted(report['graphs'], key=lambda g: report['graphs'][g]['selection_proxy'], reverse=True)
    best = ranked[0]
    improvement = report['graphs'][best]['selection_proxy'] - report['graphs']['G0']['selection_proxy']
    stress_ok = json.loads((out / 'stress60.json').read_text())['completed']
    eligible = report['complete'] and stress_ok and len(heldout_groups) >= 20 and improvement >= .01
    report['selection'] = {'graph': best if eligible else 'G0', 'best_observed_graph': best,
        'proxy_improvement_over_G0': improvement, 'new_submission_supported': eligible,
        'reason': 'complete_pilot_and_proxy_gain_at_least_0.01' if eligible else 'retain_reference_without_sufficient_evidence'}
    records = [json.loads(p.read_text()) for p in (out / 'cache').glob('*.json')]
    timings = [r['seconds'] for r in records]
    report['runtime'] = {'unique_cached_files': len(records), 'total_extraction_seconds': sum(timings),
        'mean_seconds_per_file': float(np.mean(timings)),
        'max_allocated_gib': max(r['max_allocated_gib'] for r in records),
        'max_reserved_gib': max(r['max_reserved_gib'] for r in records),
        'L4_60_minute_compliance': 'unverified; A100 measurements are not L4 evidence'}
    report['runtime']['stress60'] = [
        {k: v for k, v in json.loads(p.read_text()).items() if k != 'feature'}
        for p in (out / 'stress_cache').glob('*.json')]
    atomic_json(out / 'comparison.json', report)
    for name in predictions:
        dummy_rows = [graph_predictions(f, fusion)[name] for f in dummy_features]
        write_csv(out / f'{name}_dummy_submission.csv', dummy_rows)
    atomic_json(out / 'status.json', {'status': 'comparison_finished', 'unix': time.time(),
        'selected_graph': report['selection']['graph'], 'completed': len(features)})
    print('COMPARISON_FINISHED', json.dumps(report['selection']), flush=True)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/dacon_full_refit/package_submission.py
"""Build a selected graph's offline archive without recompressing model tensors.

The evaluator already supplies the dependencies used here, so requirements.txt
is empty. Colab setup belongs to the experiment notebook, not the submission.
"""

import csv
import json
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path, PurePosixPath
from typing import TYPE_CHECKING, Final, Literal, TypedDict

from graph_scoring import PredictionRow, SUBMISSION_COLUMNS, validate_predictions

if TYPE_CHECKING:
    from graph_fusion import Fusion


class Selection(TypedDict):
    graph: Literal["G0", "G1", "G2", "G3", "G4"]
    fusion: "Fusion | None"


RUNTIME_FILES: Final = ("graph_features.py", "graph_fusion.py", "graph_scoring.py")
MAX_ZIP_BYTES: Final = 10_000_000_000
MAX_EXPANDED_BYTES: Final = 32_000_000_000
DEPLOY_SCRIPT: Final = '''#!/usr/bin/env python3
"""Run the selected frozen graph independently on each evaluation audio file."""
import json
import os
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent
RUNTIME = ROOT / "model" / "runtime"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
sys.dont_write_bytecode = True
sys.path.insert(0, str(RUNTIME))

from graph_features import Extractor
from graph_fusion import graph_predictions
from graph_scoring import SUBMISSION_COLUMNS, validate_predictions


def main():
    graph = json.loads((RUNTIME / "choice.json").read_text())["graph"]
    if graph not in {"G0", "G1", "G2", "G3", "G4"}:
        raise ValueError("Unsupported graph selection")
    fusion = json.loads((RUNTIME / "fusion.json").read_text())
    extractor = Extractor(RUNTIME / "official_baseline.py", include_raw=graph in {"G3", "G4"})
    try:
        base = extractor.base
        columns, sample = base.read_sample_submission(ROOT / "data" / "sample_submission.csv")
        if columns != list(SUBMISSION_COLUMNS):
            raise ValueError("Sample submission columns differ from the official six-column schema")
        paths = base.order_audio_files(base.find_audio_files(ROOT / "data" / "test"), sample)
        rows = []
        for index, path in enumerate(paths, 1):
            prediction = graph_predictions(extractor.extract(path), fusion)[graph]
            rows.extend(validate_predictions([prediction], [path.stem]))
            print(f"PREDICT {index}/{len(paths)} {path.stem}", flush=True)
        rows = validate_predictions(rows, [row["ID"] for row in sample])
        destination = ROOT / "output" / "submission.csv"
        temporary = destination.with_suffix(".csv.tmp")
        base.save_submission(temporary, columns, rows)
        temporary.replace(destination)
    finally:
        extractor.close()


if __name__ == "__main__":
    main()
'''


def _model_members(archive: zipfile.ZipFile) -> list[zipfile.ZipInfo]:
    members = [member for member in archive.infolist() if member.filename.startswith("model/")]
    names = [member.filename for member in members]
    if not members or len(names) != len(set(names)):
        message = "Baseline model assets must be present with unique archive names"
        raise ValueError(message)
    for member in members:
        path = PurePosixPath(member.filename)
        if path.is_absolute() or ".." in path.parts or "\\" in member.filename:
            message = f"Unsafe baseline member: {member.filename}"
            raise ValueError(message)
        if path.parts[:2] == ("model", "runtime"):
            message = "Baseline already contains reserved model/runtime paths"
            raise ValueError(message)
    return members


def build_submission(original_zip: Path, output_zip: Path, selection: Selection) -> Path:
    """Keep baseline model assets and package one selected graph atomically."""
    if original_zip.resolve() == output_zip.resolve():
        message = "The official baseline archive must not be overwritten"
        raise ValueError(message)
    graph = selection["graph"]
    if graph not in {"G0", "G1", "G2", "G3", "G4"}:
        message = f"Unsupported graph selection: {graph}"
        raise ValueError(message)
    fusion = selection.get("fusion")
    if graph == "G4" and fusion is None:
        message = "G4 packaging requires its frozen fusion parameters"
        raise ValueError(message)
    # G0-G3 cannot accidentally execute unrelated learned-head parameters.
    fusion_json = json.dumps(fusion if graph == "G4" else None, allow_nan=False)
    source_directory = Path(__file__).resolve().parent
    runtime = {f"model/runtime/{name}": (source_directory / name).read_bytes() for name in RUNTIME_FILES}
    runtime["model/runtime/fusion.json"] = fusion_json.encode()
    runtime["model/runtime/choice.json"] = json.dumps({"graph": graph}).encode()
    compile(DEPLOY_SCRIPT, "script.py", "exec")
    output_zip.parent.mkdir(parents=True, exist_ok=True)
    temporary = output_zip.with_name(output_zip.name + ".tmp")
    try:
        with zipfile.ZipFile(original_zip) as source:
            members = _model_members(source)
            baseline = source.read("script.py").decode("utf-8")
            original_base = "BASE_DIR = Path(__file__).resolve().parent"
            if baseline.count(original_base) != 1:
                message = "Official baseline BASE_DIR no longer matches the expected source"
                raise ValueError(message)
            baseline = baseline.replace(original_base, "BASE_DIR = Path(__file__).resolve().parents[2]", 1)
            compile(baseline, "model/runtime/official_baseline.py", "exec")
            runtime["model/runtime/official_baseline.py"] = baseline.encode()
            expanded_bytes = sum(member.file_size for member in members)
            expanded_bytes += sum(map(len, runtime.values())) + len(DEPLOY_SCRIPT.encode())
            if expanded_bytes > MAX_EXPANDED_BYTES or expanded_bytes + 1_000_000 > MAX_ZIP_BYTES:
                message = f"Stored archive would exceed submission limits: {expanded_bytes} bytes"
                raise ValueError(message)
            # Tensor compression costs CPU time without improving the inference result.
            with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as target:
                for member in members:
                    if member.is_dir():
                        target.writestr(member.filename, b"")
                        continue
                    with source.open(member) as incoming, target.open(member.filename, "w", force_zip64=True) as outgoing:
                        shutil.copyfileobj(incoming, outgoing, length=8 * 1024 * 1024)
                for name, content in runtime.items():
                    target.writestr(name, content)
                target.writestr("script.py", DEPLOY_SCRIPT)
                target.writestr("requirements.txt", b"")
        if temporary.stat().st_size > MAX_ZIP_BYTES:
            message = "Built archive exceeds the compressed submission size limit"
            raise ValueError(message)
        temporary.replace(output_zip)
    finally:
        temporary.unlink(missing_ok=True)
    return output_zip


def verify_packaged(unpacked_dir: Path, deadline: float) -> list[PredictionRow]:
    """Run the real freshly unpacked script and parse its validated predictions.

    The caller provides data/test and data/sample_submission.csv, and compares
    these rows with the notebook's reference predictions. Deadline is Unix time.
    """
    remaining = deadline - time.time()
    if remaining <= 0:
        message = "No experiment time remains for packaged inference"
        raise TimeoutError(message)
    output = unpacked_dir / "output"
    output.mkdir(parents=True, exist_ok=True)
    (output / "submission.csv").unlink(missing_ok=True)
    with (output / "packaged_inference.log").open("w", encoding="utf-8") as log:
        subprocess.run(
            [sys.executable, str((unpacked_dir / "script.py").resolve())],
            cwd=unpacked_dir, stdout=log, stderr=subprocess.STDOUT,
            check=True, timeout=min(remaining, 3600),
        )
    with (unpacked_dir / "data" / "sample_submission.csv").open(encoding="utf-8-sig", newline="") as stream:
        expected_ids = [row["ID"] for row in csv.DictReader(stream)]
    with (output / "submission.csv").open(encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != list(SUBMISSION_COLUMNS):
            message = "Packaged output must have the exact official six-column header"
            raise ValueError(message)
        rows = list(reader)
    return validate_predictions(rows, expected_ids)


In [ ]:
%%writefile /content/dacon_full_refit/package_v2_submission.py
"""Create a minimal self-contained AST-only submission archive."""

from __future__ import annotations

import json
import zipfile
from pathlib import Path
from typing import Final

from package_submission import MAX_EXPANDED_BYTES, MAX_ZIP_BYTES

CHECKPOINT_FILES: Final = (
    "config.json",
    "preprocessor_config.json",
    "multitask_config.json",
    "model.safetensors",
)
RUNTIME_FILES: Final = ("ast_multitask.py", "v2_inference.py")
MODEL_INFO: Final = {
    "base_model": "MIT/ast-finetuned-audioset-10-10-0.4593",
    "revision": "f826b80d28226b62986cc218e5cec390b1096902",
    "license": "BSD-3-Clause",
    "license_source": "https://github.com/YuanGongND/ast/blob/31088be8a3f6ef96416145c4b8d43c81f99eba7a/LICENSE",
    "format": "dacon236749-ast-multitask-v1",
}
DEPLOY_SCRIPT: Final = '''#!/usr/bin/env python3
"""Run the self-contained v2 AST model with no network access."""
import os
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
sys.dont_write_bytecode = True
sys.path.insert(0, str(ROOT / "model" / "runtime"))

from v2_inference import generate_submission

if __name__ == "__main__":
    generate_submission(ROOT)
'''


def _checkpoint_members(directory: Path) -> dict[str, Path]:
    """Require every inference asset before building an archive or touching the network."""
    members = {name: directory / name for name in CHECKPOINT_FILES}
    missing = [name for name, path in members.items() if not path.is_file()]
    if missing:
        raise ValueError(f"AST checkpoint missing required assets: {', '.join(missing)}")
    try:
        metadata = json.loads(members["multitask_config.json"].read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError) as error:
        raise ValueError("AST checkpoint metadata is not valid JSON") from error
    if metadata.get("format_version") != 1 or metadata.get("mode") not in {"frozen", "last_block"}:
        raise ValueError("AST checkpoint metadata has an unsupported contract")
    return members


def build_ast_submission(checkpoint_dir: Path, output_zip: Path) -> Path:
    """Package only AST weights and local runtime files, without baseline model copies."""
    checkpoint_dir = checkpoint_dir.resolve()
    output_zip = output_zip.resolve()
    members = _checkpoint_members(checkpoint_dir)
    source = Path(__file__).resolve().parent
    runtime = {name: source / name for name in RUNTIME_FILES}
    license_text = (source / "AST_LICENSE.txt").read_bytes()
    missing_runtime = [name for name, path in runtime.items() if not path.is_file()]
    if missing_runtime:
        raise ValueError(f"Submission runtime missing: {', '.join(missing_runtime)}")
    expanded_size = sum(path.stat().st_size for path in members.values())
    expanded_size += sum(path.stat().st_size for path in runtime.values()) + len(DEPLOY_SCRIPT.encode())
    expanded_size += len(json.dumps(MODEL_INFO, sort_keys=True).encode()) + len(license_text)
    if expanded_size > MAX_EXPANDED_BYTES or expanded_size > MAX_ZIP_BYTES:
        raise ValueError(f"AST archive would exceed submission limits: {expanded_size} bytes")
    output_zip.parent.mkdir(parents=True, exist_ok=True)
    temporary = output_zip.with_name(output_zip.name + ".tmp")
    try:
        with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as archive:
            for name, path in members.items():
                archive.write(path, f"model/ast/{name}")
            archive.writestr("model/ast/MODEL_INFO.json", json.dumps(MODEL_INFO, sort_keys=True))
            archive.writestr("model/ast/LICENSE.txt", license_text)
            for name, path in runtime.items():
                archive.write(path, f"model/runtime/{name}")
            archive.writestr("script.py", DEPLOY_SCRIPT)
            archive.writestr("requirements.txt", "")
        if temporary.stat().st_size > MAX_ZIP_BYTES:
            raise ValueError("Built AST archive exceeds compressed submission size limit")
        temporary.replace(output_zip)
    finally:
        temporary.unlink(missing_ok=True)
    return output_zip


In [ ]:
%%writefile /content/dacon_full_refit/AST_LICENSE.txt
BSD 3-Clause License

Copyright (c) 2021, Yuan Gong
All rights reserved.

Redistribution and use in source and binary forms, with or without
modification, are permitted provided that the following conditions are met:

1. Redistributions of source code must retain the above copyright notice, this
   list of conditions and the following disclaimer.

2. Redistributions in binary form must reproduce the above copyright notice,
   this list of conditions and the following disclaimer in the documentation
   and/or other materials provided with the distribution.

3. Neither the name of the copyright holder nor the names of its
   contributors may be used to endorse or promote products derived from
   this software without specific prior written permission.

THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS"
AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE
IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE
DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE
FOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL
DAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR
SERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER
CAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY,
OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE
OF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.

In [ ]:
ARCHIVES = [{'sha256': '54c227b873cc3c424579fa06e8a06c442b91663de5b740b182f7cac30a3fac75', 'bytes': 346433653, 'model_revision': 'f826b80d28226b62986cc218e5cec390b1096902', 'name': 'v2_ast_assets.zip', 'parts': [{'name': 'v2_ast_assets.zip.part00', 'bytes': 83886080, 'sha256': 'f4f095b3c6062b341f6f2554c210254170da87281ee1eb547b4b6a09940764dc'}, {'name': 'v2_ast_assets.zip.part01', 'bytes': 83886080, 'sha256': 'd277ef91dcef6fe298c48ce23bbb3289b7b49e3748bd5fae2cf614f91c0c2e4d'}, {'name': 'v2_ast_assets.zip.part02', 'bytes': 83886080, 'sha256': 'df38da76de6f510582055ba33225502e7b9fa1747f487742f3dd762b12e629fb'}, {'name': 'v2_ast_assets.zip.part03', 'bytes': 83886080, 'sha256': 'b575be13671d5c3ffbc0ba74cf8a4c701c91d9270c0bffd473a7a105a623f96a'}, {'name': 'v2_ast_assets.zip.part04', 'bytes': 10889333, 'sha256': 'f2ade7b797f0ad384c0632b3d0388d4df01a9e70db6465325ac4bf9e8c2d51c3'}]}, {'name': 'v2_validation.zip', 'sha256': 'f08eef8f309fab3a176a89fe34fffeaf687fc99ffe4f16c60157addaf9796b9e', 'bytes': 217713960, 'parts': [{'name': 'v2_validation.zip.part00', 'sha256': '71ad85610171ddbab1f37bd6a20f13c6adfa5886b44d408a8983f645520876b0', 'bytes': 83886080, 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v2_validation.zip.part00'}, {'name': 'v2_validation.zip.part01', 'sha256': 'a357bb091f4e5e4302e1c5b4ce33492850d25714d2c6d66593b80a5ca1e97a04', 'bytes': 83886080, 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v2_validation.zip.part01'}, {'name': 'v2_validation.zip.part02', 'sha256': '899c0f4a3352da01d642417711ca8575c7328e1b6ec7072737edc0415a942444', 'bytes': 49941800, 'path': '/Users/choiwuseck/Documents/ChatGPT/데이콘/_workspace/v2_validation.zip.part02'}]}]

In [ ]:
OUTPUT = WORK / 'run'
try:
    with (RUN/'install.log').open('w') as log:
        subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.57.6',
            'scikit-learn==1.8.0','soundfile==0.12.1','safetensors'],
            stdout=log,stderr=subprocess.STDOUT,check=True,timeout=600)
    for archive in ARCHIVES:
        packed_path = WORK/archive['name']
        with packed_path.open('wb') as target:
            for part in archive['parts']:
                with (DRIVE/part['name']).open('rb') as source:
                    assert hashlib.file_digest(source,'sha256').hexdigest()==part['sha256']
                    source.seek(0)
                    shutil.copyfileobj(source,target,8*1024**2)
        with packed_path.open('rb') as source:
            assert hashlib.file_digest(source,'sha256').hexdigest()==archive['sha256']
        with zipfile.ZipFile(packed_path) as packed:
            packed.extractall(WORK)
    assets = next(p.parent for p in WORK.rglob('model.safetensors') if p.parent.name=='ast')
    (RUN/'code').mkdir()
    for source in WORK.iterdir():
        if source.suffix in {'.py','.txt'}:
            shutil.copy2(source,RUN/'code'/source.name)
    command = [sys.executable,'-u',str(WORK/'run_full_refit.py'),
        '--assets',str(assets),'--data',str(WORK/'v2_validation'),
        '--output',str(OUTPUT),'--device','cuda','--seconds',str(int(DEADLINE-time.time()-120))]
    with (RUN/'experiment.log').open('w') as log:
        process = subprocess.Popen(command,cwd=WORK,stdout=log,stderr=subprocess.STDOUT,env={**os.environ,'USE_TF':'0'})
        while process.poll() is None:
            if time.time()>DEADLINE-90:
                process.terminate()
                process.wait(timeout=30)
                raise TimeoutError('full-refit runtime budget reached')
            time.sleep(15)
            print((RUN/'experiment.log').read_text().splitlines()[-1:] or ['Preparing...'],flush=True)
        if process.returncode:
            raise RuntimeError((RUN/'experiment.log').read_text()[-8000:])
    print(json.loads((OUTPUT/'finished.json').read_text()))
except Exception:
    (RUN/'failure.txt').write_text(traceback.format_exc())
    raise
finally:
    try:
        if OUTPUT.exists():
            for source in OUTPUT.iterdir():
                if source.is_file():
                    shutil.copy2(source,RUN/source.name)
                elif source.name in {'warmup','finetune','final_checkpoint'}:
                    shutil.copytree(source,RUN/source.name,ignore=shutil.ignore_patterns('cache'))
            for source in OUTPUT.glob('*.zip'):
                with source.open('rb') as a,(RUN/source.name).open('rb') as b:
                    assert hashlib.file_digest(a,'sha256').hexdigest()==hashlib.file_digest(b,'sha256').hexdigest()
        (RUN/'release_requested.json').write_text(json.dumps({'elapsed_seconds':time.time()-START}))
    finally:
        timer.cancel()
        runtime.unassign()